# 본실험 세션 M4: 주 실험 N 1,200 × 시드 0·1·2

- 누가: 계정 B
- 실행 12개, 예상 약 9.9시간 (T4 x2). 시간 예산 10.5시간 안에 끝날 실행만 시작하고, 못 한 실행은 `leftover_M4.json`에 남깁니다.
- 코드: GitHub `prereg-v1` (수정할 것 없음)

**시작 전 설정 (오른쪽 패널 Session options)**
1. Accelerator: **GPU T4 x2**
2. Internet: **On**
3. Environment: **Pin to original environment**
4. Input: 없음 (이전 세션 Output을 추가하면 그 세션의 남은 실행도 먼저 처리합니다)

**실행**: 오른쪽 위 **Save Version → Save & Run All (Commit)**. 브라우저를 닫아도 끝까지 돕니다.

**끝나면**: 노트북 → Output 탭에서 **`session_M4.zip`** 을 내려받아 전달합니다.
맨 아래 셀의 요약(완료·실패·남은 실행)도 캡처해서 함께 보냅니다.

## 1. 설정 + 코드 불러오기 + GPU 확인

In [ ]:
SESSION = 'M4'
CODE_REF = 'prereg-v1'
REPO_URL = 'https://github.com/naha0106-tech/hard-negative-mllm.git'

import os, time, subprocess
os.environ['HN_T0'] = str(time.time())          # 시간 예산 계산의 시작점
REPO = '/tmp/hard-negative-mllm'                # 코드는 Output에 섞이지 않게 /tmp에
RUNS = '/kaggle/working/runs'                   # 결과는 Output으로 저장되는 곳
os.environ.update({'HN_RUNS_DIR': RUNS, 'USE_TF': '0', 'TF_CPP_MIN_LOG_LEVEL': '3',
                   'TOKENIZERS_PARALLELISM': 'false', 'HF_HUB_DISABLE_PROGRESS_BARS': '1'})
os.makedirs(RUNS, exist_ok=True)

!rm -rf {REPO} && git clone -q {REPO_URL} {REPO}
%cd {REPO}
!git checkout -q {CODE_REF}
!git log --oneline -1
gpus = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
print(gpus)
assert gpus.count('Tesla T4') >= 1, 'T4 GPU가 아닙니다. Accelerator를 GPU T4 x2로 바꾸세요.'

## 2. 설치 + 모델·데이터셋 미리 내려받기 (약 10~15분)

In [ ]:
!pip install -q transformers==4.51.3 peft==0.15.2 accelerate datasets pyarrow pyyaml 2>&1 | grep -v "dependency resolver\|requires\|incompatible" || true
!pip freeze | grep -E "^(torch|transformers|peft|accelerate|datasets)=="
!python scripts/05_evaluate.py --prefetch

## 3. 세션 실행 (학습 → 평가, GPU 2개 동시)
환경 검사 → 이전 결과 가져오기 → 실행 → 실행이 끝날 때마다 `session_M4.zip` 갱신.

In [ ]:
!python scripts/run_queue.py --session {SESSION}

## 4. 결과 요약

In [ ]:
import json, pandas as pd
p = f'/kaggle/working/session_results_{SESSION}.csv'
if os.path.exists(p):
    df = pd.read_csv(p)
    cols = [c for c in ['run_id', 'status', 'best_val_acc', 'test_acc', 'test_acc_perm', 'ai2d_acc',
                        'pope_random_f1', 'pope_adversarial_f1', 'diverged', 'minutes'] if c in df.columns]
    pd.set_option('display.width', 250)
    display(df[cols])
    print('완료', (df.status == 'done').sum(), '/', len(df))
lp = f'/kaggle/working/leftover_{SESSION}.json'
if os.path.exists(lp):
    left = json.load(open(lp))['jobs']
    print('남은 실행:', len(left), left)
print('내려받을 파일:', [f for f in os.listdir('/kaggle/working') if f.endswith('.zip')])